> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 05. Ollama 기본 실행과 요청 설정 — 60분

이 자료는 AI를 처음 배우는 비전공 학부생을 기준으로 작성했습니다. 수식과 용어가 한 번에 이해되지 않아도 괜찮습니다. 먼저 **입력 → 계산 → 출력 → 결과 해석** 의 흐름을 따라 실행하고, 수업 뒤 상세 설명을 다시 읽으세요.

본인 실습 컨테이너의 Python 커널에서 환경 셀부터 실행합니다. 설명 → 예제 → 파일 TODO → 학습자 검증 → 참고 정답 순서입니다.

실행 순서와 저장 위치는 [04.LLM README](README.md), 낯선 용어는 [04.LLM 용어 찾아보기](GLOSSARY.md)에서 바로 확인할 수 있습니다.

- **TODO는 JupyterLab 파일 편집기에서 `exercises/*.py`를 수정하고 저장합니다.** 노트북이 소스를 덮어쓰지 않습니다.
- 미완성 검증이 `AssertionError`로 실패하는 것은 정상입니다. 실패 메시지를 읽고 TODO를 고친 뒤 검증 셀을 다시 실행하세요.
- `load_op`는 저장된 파일을 다시 읽으므로 보통 커널을 재시작할 필요가 없습니다. 단, 일반 `import`로 읽은 모듈은 재시작 또는 reload가 필요할 수 있습니다.
- 완료한 앞 교시 TODO를 다음 훈련에서도 사용합니다. 참고 정답은 별도 `solutions/` 경로에 있으며, 강사가 안내하는 시점에 비교하고 자신의 코드로 재검증합니다.
- 표기 `[B,T,H]`는 텐서의 **shape** 입니다. `B`는 한 번에 처리하는 샘플 수(batch), `T`는 순서 길이(time/sequence), `H`는 내부 표현 크기(hidden), `V`는 token 종류 수(vocabulary)입니다.
- 컨테이너를 재생성하면 패키지가 포함된 최신 이미지가 필요하며, 소스와 `Data`는 호스트에 보존됩니다.

오류가 나면 가장 먼저 현재 노트북의 앞 셀을 순서대로 실행했는지, TODO 파일을 저장했는지, 오류가 난 텐서의 `shape`, `dtype`, `device`가 무엇인지 확인하세요.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 공용 Ollama와 이 노트북 준비

열 파일은 `/workspace/04.LLM/05_ollama_basics.ipynb`입니다. [05_ollama_basics.ipynb의 「Ollama의 client-server 구조」](05_ollama_basics.ipynb)에 기본 구조가 있으며 현재 필요한 내용은 다음과 같습니다. 개인 컨테이너의 Python/CLI가 `http://127.0.0.1:11434`의 공용 서버에 요청하고, LLM 수업 모델은 `gemma3:1b`입니다. 같은 Jetson의 컨테이너가 host network를 쓰므로 같은 주소로 접속합니다. 작은 모델의 loss·훈련 결과·checkpoint는 필요하지 않습니다.

**공용 Ollama 확인:** 첫 환경 셀에서 API와 `gemma3:1b` 모델 목록을 확인합니다. 연결되지 않거나 모델이 없으면 로컬 Setup 담당자에게 공용 서버·모델 저장소 확인을 요청합니다. 다른 수강생은 별도 서버를 만들거나 모델을 다운로드하지 않습니다. Python 표준 라이브러리로 HTTP를 호출하므로 이 노트북 자체는 CUDA 훈련용 import를 요구하지 않습니다.

입력 질문은 현재 셀에 포함되어 있고 API의 `messages`는 `role`·`content` 딕셔너리의 목록입니다. `stream=False`는 완료된 응답 하나를 받으며 생성 내용은 `result['message']['content']`에서 읽습니다. 첫 환경 셀부터 실행해 현재 커널의 `client`와 도우미를 다시 만들고, 해당 절에서 필요한 질문·이력 셀을 순서대로 실행합니다. 저장 로그는 `/workspace/Data/outputs/04.LLM/`에 남고 재부팅 후 메모리 대화 이력은 자동 복원되지 않습니다.

<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

이미 훈련된 언어 모델을 공용 서버에 요청하고 그 응답을 검증하는 실습입니다. 앞의 작은 모델 훈련과 달리 가중치를 수정하지 않습니다. 같은 모델에 어떤 근거와 출력 요구를 주었는지에 따라 답이 어떻게 달라지는지 비교합니다.

### 전체 과정에서 내 코드가 하는 일

```text
서버·모델 확인
  ↓
CLI/HTTP 요청
  ↓
옵션 TODO
  ↓
같은 조건의 네 질문
  ↓
응답·실패 저장
  ↓
근거와 형식 평가
```

생성 옵션의 타입·범위를 검사하고 질문 하나를 구체화합니다. 모델이 모르는 온도를 만들어 내지 않는지 원문 근거와 대조합니다. 응답의 길이와 사실성은 서로 다른 평가 항목입니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 긴 답이 나왔고 JSON도 저장되면 사실 검증이 끝났나요?

**설명:** 아닙니다. 요청 성공·파일 저장·형식 준수·근거 일치는 각각 확인해야 합니다. 지침에 없는 최고 온도는 수치 정답이 없으며 모른다고 처리하는지가 핵심입니다. 서버 시간과 생성 속도도 별도로 기록합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–5절의 수업 핵심 경로** → [2일차 슬라이드 36쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s031) · [배포 PDF 36쪽](../슬라이드/2일차.pdf#page=36)에서 44쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 작은 모델의 훈련 전후 예측 또는 Ollama 응답을 저장합니다.
- **단원 마지막 연결 질문:** 직접 훈련한 숫자 과제와 이미 학습된 모델 사용을 구분합니다. 제공한 작업 지침에 없는 질문에는 근거가 없다고 말하는지 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 시간·목표

서버·모델 10분 → 터미널 채팅 15분 → HTTP·옵션 TODO 15분 → 프롬프트 비교 15분 → 기록 5분. 앞선 훈련 모델과 Gemma는 별개입니다.

### 수업 진행 경로

1. Ollama 전용 환경 셀로 서버 연결과 gemma3:1b 목록만 확인합니다. CUDA 훈련 환경은 요구하지 않습니다.
2. 1–3절에서 사전학습 모델과 client-server 구조를 확인하고 CLI 채팅을 실행합니다.
3. 4절 payload에서 기본 옵션을 확인한 뒤 options TODO·검증과 실제 HTTP 요청을 실행합니다.
4. 5절의 두 프롬프트를 비교하고 결과 JSON을 저장합니다.
5. 양자화·성능 해석의 긴 설명과 오류 표는 수업 후 다시 읽습니다.

각 셀의 **구분** 표시를 확인하세요. `수업 후 복습·자습` 셀은 건너뛰어도 다음 수업 핵심 셀을 실행할 수 있습니다.

In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import json, subprocess, sys, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '04.LLM/common').is_dir())
COURSE = ROOT / '04.LLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import DATA, OUTPUTS
from common.loading import load_op
from common.ollama_client import OllamaClient, Conversation
OUTPUTS.mkdir(parents=True, exist_ok=True)
probe = OllamaClient()
available = [m['name'] for m in probe.models().get('models', [])]
print('Python:', sys.executable, 'Ollama:', probe.base_url, 'models:', available)
assert 'gemma3:1b' in available, 'Ollama 연결과 gemma3:1b 사전 배포를 확인하세요.'


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 지금부터는 사전학습 모델을 추론에 사용합니다

1–4번 노트북에서는 작은 모델의 파라미터를 Jetson에서 직접 바꿨습니다. 지금부터 사용하는 **Gemma 3 1B** 는 이미 대규모 데이터로 사전학습되고 지시를 따르도록 조정된 모델입니다. 이 시간에는 파라미터를 훈련하거나 미세조정하지 않고, 프롬프트를 입력해 다음 token을 생성하는 **추론** 만 수행합니다.

이 차이를 분명히 구분하세요.

| 작업 | 파라미터 변경 | 예 |
|---|---:|---|
| 모델 파일 다운로드/pull | 아니요 | 준비된 가중치 파일을 저장장치에 복사 |
| 모델 메모리 적재 | 아니요 | 추론을 위해 가중치를 RAM/GPU에서 사용할 준비 |
| 추론 | 아니요 | 프롬프트에 대한 token 생성 |
| 미세조정/훈련 | 예 | gradient와 optimizer로 가중치 업데이트 |

이 모델의 약 1B는 대략 10억 개 규모의 파라미터를 뜻합니다. `Q4_K_M` 같은 표기는 많은 가중치를 약 4-bit 수준으로 표현한 **양자화(quantization)** 형식입니다. 양자화는 모델 파일과 메모리 사용량을 줄이고 엣지 장치 추론을 가능하게 하지만, 원래 정밀도와 완전히 동일한 계산은 아니며 품질·속도·메모리 사이에 trade-off가 있습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. Ollama의 client-server 구조

<!-- procedure:lesson-15-7dffa3:start -->
![공용 모델 서버에 요청하기](../교재그림/절차/lesson-15.svg)

클라이언트는 개인별로, 서버·모델 저장소는 공유합니다. 모델 tag와 실제 준비 상태를 먼저 확인합니다.
<!-- procedure:lesson-15-7dffa3:end -->


Ollama는 모델을 관리하고 추론 API를 제공하는 프로그램입니다. 이번 환경은 역할을 두 컨테이너로 나눕니다.

```text
Jupyter/터미널(공통 실습 컨테이너)
        │ HTTP 요청: 127.0.0.1:11434
        ▼
Ollama 서버(별도 컨테이너) ── Data/ollama의 모델 파일 사용
        │
        ▼
Jetson GPU/공유 메모리에서 추론
```

- 지금 보고 있는 노트북과 `ollama` CLI는 **client** 입니다.
- `physical_ai_ollama` 컨테이너가 **server** 입니다.
- 두 컨테이너가 host network를 사용하므로 같은 Jetson에서 `127.0.0.1:11434`로 연결됩니다.
- `127.0.0.1`은 각 장치 자신을 가리킵니다. 다른 Jetson의 주소가 아닙니다.
- 모델 파일은 호스트 `Data/ollama`에 미리 저장되어 컨테이너가 재생성되어도 남습니다.

수강생마다 모델을 pull하거나 Ollama 서버를 새로 띄우지 않습니다. 같은 Jetson의 사용자는 한 서버와 GPU 자원을 공유합니다. 지금 사용하는 Gemma는 텍스트 채팅 경로이며, 다음 VLM 시간의 이미지 입력 실습과 구분합니다.


In [ ]:
# 구분: 수업 핵심 셀
client=OllamaClient()
model_list=client.models()
for model_info in model_list['models']:
    print('이름:', model_info['name'])
    print('파일 크기(byte):', model_info['size'])
    print('digest:', model_info['digest'])
assert any(m['name']=='gemma3:1b' for m in model_list['models']), '모델 사전 배포를 확인하세요.'

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

### 모델 이름, tag, digest

`gemma3:1b`에서 `gemma3`는 모델 계열 이름이고 `1b`는 선택한 tag입니다. tag는 사람이 쓰기 쉬운 이름이라 배포 시점에 가리키는 내용이 달라질 여지가 있습니다. **digest** 는 실제 모델 내용에 대응하는 식별값이므로, 수업 환경 재현성은 이름과 digest를 함께 기록합니다.

`ollama list`에 보인다는 것은 모델 파일이 서버 저장소에 등록되어 있다는 뜻입니다. 모델이 지금 GPU 메모리에 계속 적재되어 있다는 뜻은 아닙니다. 첫 요청이 느리고 다음 요청이 빠를 수 있는 이유 중 하나가 최초 적재입니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 터미널에서 대화 (15분)

JupyterLab의 Terminal을 열어 다음을 실행합니다. 이 명령은 공통 실습 컨테이너에 추가된 CLI이며 별도 서버에 접속합니다.

```bash
ollama list
ollama run gemma3:1b
```

`공장의 온도 센서가 하는 일을 두 문장으로 설명해줘.`를 입력하세요. 이어서 `한 문장으로 줄여줘.`를 입력하고 문맥 사용을 관찰합니다. `/bye`로 종료합니다.
두 번째 요청이 첫 답변을 줄일 수 있는 이유는 같은 실행 중 CLI가 이전 메시지를 함께 보내기 때문입니다. `/bye`로 종료한 뒤 다시 시작해 `앞 답을 한 문장으로 줄여줘.`만 입력하고 비교하세요. 이전 프로세스의 이력이 자동 복원된다고 가정하지 말고, 실제로 전달된 세션 범위를 관찰합니다. 영구 대화 저장은 애플리케이션이 별도로 구현해야 합니다.

모델의 문장은 정해진 정답 문자열이 아닙니다. 사실 관계, 요청한 문장 수, 불필요한 내용 여부를 사람이 확인합니다. 자연스러운 문장과 사실인 문장은 같은 뜻이 아닙니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 4. Chat API의 요청과 응답

`POST /api/chat`에는 model과 role/content 메시지 목록을 보냅니다. `stream=False`를 사용해 응답 전체를 JSON 하나로 받습니다.
서버의 token 생성 속도는 `eval_count / (eval_duration / 1e9)`로 계산합니다. 첫 요청의 모델 로드 시간은 별도이므로 전체 wall time과 구분합니다.
`num_ctx`는 문맥 길이 설정, `num_predict`는 출력 token 상한입니다. token 수와 한글 글자 수가 같지는 않습니다.

현재 `OllamaClient.chat(..., num_predict=64)`가 만드는 대표 실제 payload의 기본값은 다음과 같습니다. `options` TODO가 만드는 dict는 temperature/num_ctx/num_predict 부분이며, client가 기본 `seed`와 바깥의 `keep_alive`를 합쳐 전송합니다.

```json
{
  "model": "gemma3:1b",
  "messages": [
    {"role": "user", "content": "온도 센서의 역할은?"}
  ],
  "stream": false,
  "options": {
    "temperature": 0,
    "seed": 2026,
    "num_ctx": 2048,
    "num_predict": 64
  },
  "keep_alive": "5m"
}
```

- `seed=2026`은 sampling에 사용하는 난수 시작값을 고정해 같은 환경에서 응답 비교의 변동을 줄입니다. 작은 모델 훈련 seed와 숫자는 같지만 별개의 설정이며, 모델·버전·하드웨어가 다른 경우까지 동일 문자열을 보장하지 않습니다.
- `keep_alive="5m"`은 요청이 끝난 뒤 모델을 약 5분 동안 메모리에 유지하도록 서버에 요청합니다. 다음 요청의 재적재 시간을 줄일 수 있습니다. 대화 이력을 저장하는 시간이나 모델 파일 보존 기간이 아니며, 서버 자원 상황과 구현에 따라 실제 적재 상태는 달라질 수 있습니다.

메시지의 `role`은 대화에서의 역할을 나타냅니다.

- `system`: 응답의 전반적인 역할과 규칙을 제시할 때 사용
- `user`: 사용자의 질문이나 지시
- `assistant`: 모델이 앞에서 생성한 응답; 대화 이력을 다시 보낼 때 포함

응답에는 생성된 `message.content` 외에도 수행 시간과 token 수 관련 필드가 들어올 수 있습니다. 시간 값 일부는 nanosecond 단위이므로 초로 바꿀 때 `1e9`로 나눕니다. `eval_count`는 생성한 출력 token 수이며 한글 글자 수, 어절 수, 문장 수와 같지 않습니다.

`stream=False`는 생성이 끝난 뒤 JSON 하나를 받아 입문 실습에서 다루기 쉽습니다. `stream=True`는 생성 중 조각을 연속으로 받아 화면에 빨리 보여 줄 수 있지만, 조각 결합·종료·오류 처리가 추가로 필요합니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 수업 핵심 TODO — options

수정 파일: **[04.LLM/exercises/options.py](exercises/options.py)**

temperature=0, num_ctx=2048, num_predict=인자 값인 dict를 만드세요. 출력 token 상한은 양의 정수이며 잘못된 입력에는 ValueError를 발생시킵니다.
설정을 직접 만들어 다음 실제 요청에 사용합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `build_options(num_predict)`는 bool이 아닌 양의 정수를 받아 `{'temperature': 0, 'num_ctx': 2048, 'num_predict': num_predict}`를 반환합니다. `build_options(64)`의 출력에는 64가 들어갑니다.
**작성 흐름 힌트:** 타입 검사에서 bool 제외 → 1 이상인지 확인 → 세 key의 새 dict 반환. `0`, `-1`, `1.5`, `'64'`, `True`는 모두 ValueError입니다. 서버 호출이나 모델 다운로드는 이 함수의 일이 아닙니다.
**설명할 지점:** “num_ctx는 문맥 크기, num_predict는 출력 토큰 상한입니다. 답변 글자 수를 정확히 지정하는 값은 아닙니다.”
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE / 'exercises/options.py').read_text())

In [ ]:
# 구분: 수업 핵심 셀
from tests.test_ollama import check_options
check_options('exercise')
print('learner_verification=PASS')

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

### 참고 정답

자습에서는 필요할 때 정답과 해설을 열어 비교하고 자신의 코드로 재검증합니다. 아래 실행은 자신의 TODO를 변경하지 않습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'solutions/options.py').read_text())

### 정답 해설 — 입력부터 결과까지

**옵션의 역할:** 정답은 num_predict가 bool이 아닌 양의 정수인지 먼저 검사하고 temperature=0, num_ctx=2048과 함께 새 사전을 만듭니다. Python에서 bool은 int의 하위 타입이라 `isinstance(True,int)`만으로는 잘못 통과할 수 있습니다.

num_predict=32는 출력 token 상한을 32로 정하는 것이며 32글자를 보장하지 않습니다. 0·음수·True는 오류입니다. temperature=0도 모든 환경·모델에서 같은 문장을 보장한다는 뜻은 아닙니다.

**확인 질문:** 짧게 끊긴 답을 보았을 때 모델의 지식 부족과 출력 상한을 어떻게 구분하나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/options.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
def build_options(num_predict):
    # bool은 int의 하위 타입이지만 토큰 개수로 허용하지 않습니다.
    if not isinstance(num_predict,int) or isinstance(num_predict,bool) or num_predict<1:
        raise ValueError('num_predict는 양의 정수여야 합니다.')
    # 요청 옵션만 구성합니다. 실제 모델 호출은 client가 수행합니다.
    return {'temperature':0,'num_ctx':2048,'num_predict':num_predict}
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
check_options('solution')
print('solution_verification=PASS')

> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

### 생성 옵션을 정확히 해석하기

- **temperature** : 다음 token 후보 분포를 얼마나 평평하거나 뾰족하게 사용할지 조절합니다. 낮을수록 높은 점수의 token을 더 강하게 선호하는 경향이 있습니다. `0`은 비교 실습의 변동을 줄이려는 설정이지만, 모든 버전·하드웨어·설정에서 byte 단위로 같은 출력을 기술적으로 보장하는 스위치는 아닙니다.
- **num_ctx** : 모델이 한 요청에서 고려하도록 잡는 문맥 token 용량입니다. 입력 메시지와 생성에 필요한 공간이 관련됩니다. 값을 늘리면 메모리와 계산 비용이 늘 수 있으며 모델 자체가 지원하는 최대 문맥을 무한히 확장하지는 않습니다.
- **num_predict** : 새로 생성할 token 수의 상한입니다. 답의 글자 수나 문장 수를 직접 보장하지 않습니다. 너무 작으면 문장이 중간에 잘릴 수 있습니다.

`num_predict`가 양의 정수인지 검사하는 이유는 잘못된 사용자 입력을 서버에 보내기 전에 명확한 오류로 막기 위해서입니다. Python의 `bool`은 `int`의 하위형처럼 취급될 수 있어 구현과 검증에서 의도한 입력 범위를 주의해야 합니다.

### Vocabulary 크기와 context 길이는 서로 다른 값입니다

- **Vocabulary(어휘 집합) 크기** 는 모델의 tokenizer가 구분하는 token 종류 수입니다. 같은 모델·tokenizer를 사용할 때 `num_ctx`를 늘려도 token 종류가 추가되지는 않습니다.
- **Context(문맥) 길이 — `num_ctx`** 는 생성 중 한 번에 참고할 수 있는 token 시퀀스의 길이 설정입니다. 같은 token이 반복되어도 등장할 때마다 길이에 포함됩니다.
- **출력 token 상한 — `num_predict`** 는 새로 생성할 token 수의 최대값입니다. 종료 token이 먼저 나오면 더 짧게 끝나며, 글자 수와 같지 않습니다.

이전 대화가 **추가 token 입력** 이 된다는 말은 vocabulary가 커진다는 뜻이 아니라, 기존 token들로 이루어진 **입력 시퀀스가 길어진다** 는 뜻입니다. 입력에는 system 지시, 이전 user/assistant 대화, 현재 질문, 모델의 채팅 템플릿이 붙이는 구분 token도 포함됩니다.

**문맥 예산 예시:** `num_ctx=2048`, `num_predict=256`이라면, 전체 입력을 유지하면서 최대 256 token의 답변 공간을 확보하기 위해 입력을 **1,792 token 이내** 로 계획합니다.

`입력 최대 1,792 + 생성 여유 256 = 문맥 2,048 token`

이 계산은 전체 문맥을 보존하기 위한 계획 기준입니다. 실제 출력 길이가 언제나 `num_ctx − 입력 길이`로 제한된다는 보장은 아닙니다. 한도를 넘었을 때의 잘림·문맥 이동·오류 처리는 모델과 실행 환경에 따라 달라집니다. 긴 대화는 오래된 이력을 줄이거나 요약하고, 필요하면 모델 지원 범위와 메모리 여유 안에서 `num_ctx`를 늘립니다. `num_predict`만 늘려서는 context 용량이 늘어나지 않습니다.

공식 참고: [Ollama 생성 파라미터](https://docs.ollama.com/modelfile#valid-parameters-and-values), [Context 길이와 메모리](https://docs.ollama.com/context-length)



In [ ]:
# 구분: 수업 핵심 셀
options=load_op('options').build_options(64)
answer=client.chat([{'role':'user','content':'온도 센서의 역할을 한국어 한 문장으로 설명해줘.'}],options=options)
print(answer['message']['content'])
print('전체 wall seconds:',answer['wall_seconds'])
print('출력 tokens/s:',answer['tokens_per_second'])

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

### 시간과 속도 결과 읽기

**wall time** 은 client가 요청을 시작해 응답 전체를 받을 때까지 사용자가 기다린 실제 시간에 가깝습니다. 모델 로드, prompt 처리, token 생성, 통신과 직렬화가 포함될 수 있습니다. **tokens/s** 는 보통 출력 token 생성 구간의 처리량입니다. 두 값은 역수 관계가 아니며 같은 질문에서도 첫 요청, 온도·전력 모드, 다른 사용자의 GPU 부하와 출력 길이에 따라 달라집니다.

짧은 출력은 고정 준비 시간의 영향이 커서 tokens/s만 보고 모델 전체 체감 성능을 판단하기 어렵습니다. 비교할 때는 모델, 프롬프트, 출력 상한, 전력 모드와 동시 부하를 함께 기록합니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 5. 프롬프트 비교 (15분)

같은 모델에서 질문을 모호하게 쓸 때와 대상·출력 형식을 지정할 때의 차이를 관찰합니다.
수강생 TODO: 아래 두 프롬프트 중 하나를 공정 사례에 맞게 바꾸고, 길이·형식 준수·사실성을 기록하세요. 짧은 모델의 답을 검증 없이 실제 장비 제어에 연결하지 않습니다.

좋은 프롬프트에 언제나 하나의 공식이 있는 것은 아니지만, 다음 요소가 있으면 요구사항을 해석하기 쉽습니다.

1. **상황/대상** : 어떤 설비와 어떤 가정인지
2. **작업** : 설명, 분류, 추출, 요약 중 무엇을 원하는지
3. **제약** : 사용할 수 있는 정보, 하지 말아야 할 추측, 답의 범위
4. **출력 형식** : 문장 수, 목록, JSON 필드 등
5. **검증 기준** : 어떤 값이 빠지면 안 되는지

프롬프트를 자세히 쓴다고 모델의 지식이 자동으로 정확해지지는 않습니다. 모델은 그럴듯하지만 틀린 내용을 만들 수 있으며 이를 흔히 **hallucination(환각)** 이라고 부릅니다. 모델에는 센서의 현재 실제 값이나 장비 매뉴얼이 자동으로 연결되어 있지 않습니다. 제시하지 않은 최신 상태를 안다고 가정하지 마세요.

실제 제어 후보를 만들 때는 허용 명령 목록, 값 범위, 센서 기반 검증, timeout, 비상 정지와 사람 승인 같은 결정적 프로그램 경계를 별도로 둬야 합니다. 자연어 답변을 그대로 actuator 명령으로 실행하지 않습니다.

### 이 TODO의 입력·수정 범위·완료 기준

아래 `prompts`에는 모호한 질문과 공정 조건이 구체적인 질문이 있습니다. 둘 중 하나를 상황·주어진 근거·출력 형식이 분명한 질문으로 수정하고 모델과 `options`는 같게 유지합니다.

**공통 사례의 위치와 현재 필요한 내용:** [공통사례.md의 작업 지침 질문](../공통사례.md)과 2일차 슬라이드 「작업 지침에 있는 것과 없는 것」에는 다음 지침이 있습니다. **“빨간 블록과 파란 블록을 확인하고, 가려져 구분할 수 없으면 담당자 확인을 요청한다.”** 이 지침을 질문 안에 넣어 “파란 블록이 가려졌다면?”에는 담당자 확인을, “허용 최고 온도는?”에는 지침에 없음을 말하는지 검사합니다. 아래 비교 셀에도 이 지침과 두 질문을 함께 넣었으므로 다른 문서를 열 필요가 없습니다. 이 질문은 지침 읽기 실습이며 실제 사진 판정을 수행하지 않습니다.

비교 셀은 총 네 응답을 출력·저장합니다. 바로 다음 **평가 기록 TODO 셀** 에서 각 응답의 `length`(실제 길이와 충분성), `format`(요청 형식 준수 여부와 근거), `factuality`(주어진 근거와 일치하는 주장/근거 없는 주장), `unknown_handling`(모르는 정보에 대한 처리)을 채우세요. 앞의 두 질문에서 미제공 정보 질문을 하지 않았다면 `unknown_handling`에는 “해당 질문 없음”과 이유를 적을 수 있습니다. 네 번째 응답에서 모델이 제시한 임의 온도를 정답으로 받아들이지 않습니다.

평가 셀만 수정·재실행하면 같은 응답의 JSON에 기록이 추가됩니다. 비교 API 셀의 `RESUME_COMPARISON=None`은 새 비교·새 파일을 만듭니다. 기존 응답을 평가할 때는 평가 셀만 실행합니다. 자연어 답은 고정 문자열로 채점하지 않으며 네 응답의 네 평가 항목과 근거를 저장해야 완료입니다.

### 요청 일부가 실패했을 때

네 질문은 요청마다 성공·실패·소요 시간을 즉시 파일에 저장합니다. 세 번째 요청이 timeout이어도 앞의 성공한 두 응답은 남습니다. `status=error/pending` 또는 attempt의 `started`가 남으면 실패·중단 기록입니다.

아래 셀에서 출력된 **비교 기록 경로** 를 복사해 `RESUME_COMPARISON = '/workspace/Data/.../prompt-comparison-실제시각.json'`처럼 기존 파일 경로로 바꾼 뒤 같은 셀을 다시 실행합니다. 위 경로의 `...`는 그대로 쓰지 않고 실제 출력 전체를 사용합니다. 질문·모델·options가 같아야 하며 성공한 응답은 다시 요청하지 않습니다. `attempts`에는 앞 실패와 이번 재시도가 함께 남습니다.

커널을 재시작했다면 먼저 환경·import·client·options 셀을 준비하고 같은 질문을 복원합니다. 질문이나 모델 조건을 바꾸려면 `None`으로 새 비교를 만듭니다. 네 응답이 모두 성공한 뒤 아래 평가 TODO를 작성합니다. 요청 실패를 모델이 “모른다”고 답한 의미 평가로 채우지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
prompts=['온도가 높아.', '제조 설비 온도가 85도라는 가상의 상황이다. 확인할 항목 2개를 짧은 목록으로 제안해줘.']
work_instruction = '빨간 블록과 파란 블록을 확인하고, 가려져 구분할 수 없으면 담당자 확인을 요청한다.'
grounded_prompts = [
    f'작업 지침: {work_instruction} 이 지침만 근거로 답하라. 파란 블록이 가려졌다면?',
    f'작업 지침: {work_instruction} 이 지침만 근거로 답하고 없으면 모른다고 말하라. 허용 최고 온도는?',
]

# None이면 새 비교입니다. 실패 복구 시 출력된 기존 JSON 경로를 문자열로 넣습니다.
RESUME_COMPARISON = None
request_prompts = prompts + grounded_prompts
if RESUME_COMPARISON is None:
    path = OUTPUTS/f'prompt-comparison-{time.time_ns()}.json'
    comparisons = [
        {'prompt': prompt, 'model': client.model, 'options': dict(options),
         'response': None, 'status': 'pending', 'attempts': [],
         'review': {'length': '', 'format': '', 'factuality': '', 'unknown_handling': ''}}
        for prompt in request_prompts
    ]
else:
    path = Path(RESUME_COMPARISON)
    comparisons = json.loads(path.read_text(encoding='utf-8'))
    assert len(comparisons) == len(request_prompts), '같은 네 질문의 기록을 선택하세요.'
    assert [row['prompt'] for row in comparisons] == request_prompts, '질문이 바뀌면 새 비교를 만드세요.'
    assert all(row.get('model') == client.model and row.get('options') == options
               for row in comparisons), '모델·옵션이 바뀌면 새 비교를 만드세요.'

def save_comparisons():
    path.write_text(json.dumps(comparisons, ensure_ascii=False, indent=2), encoding='utf-8')

save_comparisons()
print('비교 기록:', path)
for index, entry in enumerate(comparisons, 1):
    if entry['status'] == 'success':
        print(index, '기존 성공 응답 재사용:', entry['response']['message']['content'])
        continue
    attempt = {'number': len(entry['attempts']) + 1, 'status': 'started'}
    entry['attempts'].append(attempt)
    save_comparisons()  # 도중 중단되어도 시작한 요청과 앞의 성공 결과를 보존
    started = time.perf_counter()
    try:
        result = client.chat([{'role': 'user', 'content': entry['prompt']}], options=entry['options'])
        answer = result['message']['content']
        assert isinstance(answer, str) and answer.strip(), '빈 응답 또는 잘못된 응답 구조'
    except Exception as exc:
        entry['status'] = 'error'
        attempt.update(status='error', error=f'{type(exc).__name__}: {exc}')
        print(index, '요청 실패:', attempt['error'])
    else:
        entry.update(status='success', response=result)
        attempt['status'] = 'success'
        print(entry['prompt'], '\n', answer)
    finally:
        attempt['elapsed_seconds'] = time.perf_counter() - started
        save_comparisons()
print('성공:', sum(row['status'] == 'success' for row in comparisons), '/', len(comparisons))
print('기록:', path, '— 오류/미실행이 있으면 같은 경로로 재개하세요.')


In [ ]:
# 구분: 수업 핵심 셀 — 평가 기록 TODO
# 위에서 출력한 네 응답과 실제 근거를 보고 빈 문자열을 채우세요.
# length: 실제 길이와 충분성 / format: 형식 준수 / factuality: 근거 일치
# unknown_handling: 미제공 정보를 모른다고 했는지 또는 해당 질문 없음의 이유
prompt_reviews = [
    {'length': '', 'format': '', 'factuality': '', 'unknown_handling': ''},  # 모호한 질문
    {'length': '', 'format': '', 'factuality': '', 'unknown_handling': ''},  # 구체적인 질문
    {'length': '', 'format': '', 'factuality': '', 'unknown_handling': ''},  # 가려진 파란 블록
    {'length': '', 'format': '', 'factuality': '', 'unknown_handling': ''},  # 지침에 없는 최고 온도
]
assert all(row.get('status') == 'success' for row in comparisons), '실패·미실행 요청을 복구한 뒤 네 응답을 평가하세요.'
assert len(prompt_reviews) == len(comparisons), '비교 응답마다 평가를 하나씩 작성하세요.'
for entry, review in zip(comparisons, prompt_reviews):
    entry['review'] = review
path.write_text(json.dumps(comparisons, ensure_ascii=False, indent=2), encoding='utf-8')
saved_comparisons = json.loads(path.read_text(encoding='utf-8'))
assert saved_comparisons == comparisons
review_complete = all(value.strip() for row in prompt_reviews for value in row.values())
print(path, '평가 기록 완료:', review_complete)


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 완료·자습

CLI와 HTTP 요청을 모두 실행하고 모델 이름·옵션·전체 시간·token 생성 속도의 차이를 설명합니다.
자습(20분): num_predict를 32/128로 바꿔 잘림·시간을 비교합니다. temperature=0이어도 다른 환경까지 동일 결과를 보장한다고 해석하지 않습니다.

참고: [Ollama Chat API](https://docs.ollama.com/api/chat), [Gemma 3 1B](https://ollama.com/library/gemma3:1b)


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

## 오류가 날 때 확인할 순서

| 증상 | 가능한 원인 | 확인/조치 |
|---|---|---|
| `Connection refused` | Ollama 서버가 실행 중이 아니거나 주소가 다름 | 로컬 담당자가 서버 컨테이너 상태를 확인; 수강생이 새 서버를 중복 실행하지 않음 |
| `model not found` | `Data/ollama` 연결 또는 모델 tag 불일치 | `ollama list`의 정확한 이름과 사전 배포 Data 확인 |
| 첫 응답이 오래 걸림 | 모델을 메모리에 처음 적재 중 | 첫 요청과 후속 요청 시간을 따로 기록하고 기다림 |
| 응답이 중간에 끊김 | `num_predict`가 작거나 모델이 종료 token을 생성 | 상한과 응답 메타데이터를 확인하고 목적에 맞게 조정 |
| 요청 형식 오류/HTTP 400 | JSON 필드·자료형이 잘못됨 | model, messages, options의 실제 payload 확인 |
| timeout | 긴 prompt/출력, 동시 GPU 사용, 서버 문제 | 출력 상한과 다른 사용자 작업을 확인; 무조건 timeout만 크게 하지 않음 |
| 한글 문장 수가 지시와 다름 | token 상한은 문장 수 제약이 아니며 모델은 지시를 어길 수 있음 | 출력 형식을 명시하고 응답을 프로그램으로 검증 |
| 같은 prompt인데 문장이 다름 | sampling, 버전, 실행 환경 차이 | 자연어 문자열 일치가 아닌 사실·형식 기준으로 평가 |

오류 메시지를 숨기고 package를 즉시 재설치하지 마세요. 이 수업은 검증된 이미지와 별도 서버를 사용하므로, 먼저 client 연결·서버·모델 목록·요청 payload 중 어느 단계가 문제인지 나눠 확인합니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** CLI·API 요청, options 검증, 프롬프트·지연 비교 저장

[2일차 슬라이드 36쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s031) · [배포 PDF 36쪽](../슬라이드/2일차.pdf#page=36)로 돌아가 **36–44쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

